# Pipeline preprocessing для train-данных Credit Score

Этот ноутбук переносит выводы EDA в переиспользуемый preprocessing pipeline, совместимый со sklearn.

Основные решения:
- нормализуем грязные маркеры пропусков, например `_`, `_______` и `!@9#%8`;
- преобразуем строковые числовые признаки в числа;
- преобразуем `Credit_History_Age` в количество месяцев;
- считаем невозможные значения из EDA пропусками перед заполнением;
- при необходимости используем профили по `Customer_ID`, обученные только на train-части;
- разворачиваем `Type_of_Loan` из списка через запятую в числовые индикаторы типов кредитов;
- добавляем инженерные ratio- и flag-признаки по долговой нагрузке, платежной истории и структуре кредитных продуктов;
- удаляем идентификаторы и персональные поля перед формированием модельной матрицы;
- заполняем числовые и категориальные признаки, масштабируем числовые признаки и one-hot кодируем категориальные.

In [ ]:
from pathlib import Path
import re
from typing import Iterable

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "Credit_Score"
SCORE_ORDER = ["Poor", "Standard", "Good"]
SCORE_MAP = {"Poor": 0, "Standard": 1, "Good": 2}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "train.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "train.csv"

In [ ]:
train_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(train_raw.shape)
train_raw.head()

In [ ]:
MISSING_MARKERS = {
    "",
    "_",
    "__",
    "___",
    "_______",
    "!@9#%8",
    "nan",
    "NaN",
    "None",
    "<NA>",
}

NUMERIC_LIKE_COLS = [
    "Age",
    "Annual_Income",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Outstanding_Debt",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

EXPECTED_RANGES = {
    "Age": (18, 100),
    "Annual_Income": (0, 1_000_000),
    "Monthly_Inhand_Salary": (0, 100_000),
    "Num_Bank_Accounts": (0, 20),
    "Num_Credit_Card": (0, 20),
    "Interest_Rate": (0, 100),
    "Num_of_Loan": (0, 20),
    "Delay_from_due_date": (0, 90),
    "Num_of_Delayed_Payment": (0, 100),
    "Num_Credit_Inquiries": (0, 100),
    "Outstanding_Debt": (0, 20_000),
    "Total_EMI_per_month": (0, 20_000),
    "Amount_invested_monthly": (0, 20_000),
    "Monthly_Balance": (0, 10_000),
    "Credit_History_Age_Months": (0, 600),
}

DROP_AFTER_CLEANING = [
    "ID",
    "Customer_ID",
    "Name",
    "SSN",
    "Credit_History_Age",
    "Type_of_Loan",
    TARGET,
]

PROFILE_NUMERIC_COLS = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_History_Age_Months",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

PROFILE_CATEGORICAL_COLS = [
    "Occupation",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
]

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

ENGINEERED_FEATURES = [
    "debt_to_annual_income",
    "debt_to_monthly_salary",
    "emi_to_monthly_salary",
    "investment_to_monthly_salary",
    "balance_to_monthly_salary",
    "available_income_after_emi",
    "available_income_after_emi_and_investment",
    "delayed_payment_ratio",
    "avg_delay_per_delayed_payment",
    "inquiries_per_credit_account",
    "credit_cards_per_bank_account",
    "loans_per_bank_account",
    "credit_history_years",
    "credit_age_per_loan",
    "total_credit_products",
    "loan_diversity_ratio",
    "has_negative_payment_history",
    "high_utilization_flag",
    "low_balance_flag",
    "debt_per_credit_product",
]


def normalize_missing_markers(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.mask(cleaned.isin(MISSING_MARKERS))
    return cleaned


def parse_numeric(series: pd.Series) -> pd.Series:
    cleaned = normalize_missing_markers(series)
    cleaned = cleaned.str.replace("_", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce").astype("float64")


def parse_credit_history_to_months(series: pd.Series) -> pd.Series:
    cleaned = normalize_missing_markers(series)
    parts = cleaned.str.extract(
        r"(?:(\d+)\s+Years?)?\s*(?:and\s*)?(?:(\d+)\s+Months?)?",
        expand=True,
    )
    years = pd.to_numeric(parts[0], errors="coerce")
    months = pd.to_numeric(parts[1], errors="coerce")
    parsed = years.fillna(0).mul(12).add(months.fillna(0))
    return parsed.where(parts.notna().any(axis=1))


def parse_loan_types(value: object) -> list[str]:
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text in MISSING_MARKERS:
        return []

    text = re.sub(r"\band\b", ",", text, flags=re.IGNORECASE)
    loan_types = []
    for part in text.split(","):
        cleaned = part.strip().strip(".")
        if cleaned and cleaned not in MISSING_MARKERS:
            loan_types.append(cleaned)
    return loan_types


def _series_mode(series: pd.Series) -> object:
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.mode(dropna=True).iloc[0]

In [ ]:
class CreditScoreCleaner(BaseEstimator, TransformerMixin):
    # Очищает raw-строки Credit Score перед общим sklearn preprocessing.

    def __init__(
        self,
        use_customer_profiles: bool = True,
        clip_quantiles: tuple[float, float] = (0.01, 0.99),
    ):
        self.use_customer_profiles = use_customer_profiles
        self.clip_quantiles = clip_quantiles

    def fit(self, X: pd.DataFrame, y: object = None):
        data = self._basic_clean(X)
        self.feature_columns_in_ = list(X.columns)

        loan_counts = {}
        if "Type_of_Loan" in data.columns:
            for loans in data["Type_of_Loan"].apply(parse_loan_types):
                for loan in loans:
                    loan_counts[loan] = loan_counts.get(loan, 0) + 1
        self.loan_types_ = sorted(loan_counts)

        self.numeric_columns_ = [
            column
            for column in data.select_dtypes(include=[np.number]).columns
            if column not in [TARGET]
        ]

        self.clip_bounds_ = {}
        if self.clip_quantiles is not None:
            lower_q, upper_q = self.clip_quantiles
            for column in self.numeric_columns_:
                series = pd.to_numeric(data[column], errors="coerce")
                if series.notna().any():
                    lower, upper = series.quantile([lower_q, upper_q])
                    self.clip_bounds_[column] = (float(lower), float(upper))

        if self.use_customer_profiles and "Customer_ID" in data.columns:
            self.customer_numeric_profiles_ = self._fit_customer_numeric_profiles(data)
            self.customer_categorical_profiles_ = self._fit_customer_categorical_profiles(data)
        else:
            self.customer_numeric_profiles_ = pd.DataFrame()
            self.customer_categorical_profiles_ = pd.DataFrame()

        cleaned = self._apply_customer_profiles(data)
        cleaned = self._add_loan_features(cleaned)
        cleaned = self._clip_numeric_features(cleaned)
        cleaned = self._add_credit_engineering_features(cleaned)
        cleaned = self._post_clean(cleaned)
        self.output_columns_ = list(cleaned.columns)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        data = self._basic_clean(X)
        data = self._apply_customer_profiles(data)
        data = self._add_loan_features(data)
        data = self._clip_numeric_features(data)
        data = self._add_credit_engineering_features(data)
        data = self._post_clean(data)

        for column in self.output_columns_:
            if column not in data.columns:
                data[column] = np.nan
        return data[self.output_columns_]

    def _basic_clean(self, X: pd.DataFrame) -> pd.DataFrame:
        data = X.copy()

        for column in data.select_dtypes(include=["object", "string"]).columns:
            data[column] = normalize_missing_markers(data[column])

        for column in NUMERIC_LIKE_COLS:
            if column in data.columns:
                data[column] = parse_numeric(data[column])

        if "Credit_History_Age" in data.columns:
            data["Credit_History_Age_Months"] = parse_credit_history_to_months(
                data["Credit_History_Age"]
            )

        if "Month" in data.columns:
            data["Month_Num"] = data["Month"].map(MONTH_ORDER).astype("float64")

        for column, (lower, upper) in EXPECTED_RANGES.items():
            if column in data.columns:
                series = pd.to_numeric(data[column], errors="coerce")
                data[column] = series.mask((series < lower) | (series > upper)).astype("float64")

        return data

    def _fit_customer_numeric_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        columns = [
            column
            for column in PROFILE_NUMERIC_COLS
            if column in data.columns and data[column].notna().any()
        ]
        if not columns:
            return pd.DataFrame()
        return data.groupby("Customer_ID", dropna=True)[columns].median()

    def _fit_customer_categorical_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        columns = [
            column
            for column in PROFILE_CATEGORICAL_COLS
            if column in data.columns and data[column].notna().any()
        ]
        if not columns:
            return pd.DataFrame()
        return data.groupby("Customer_ID", dropna=True)[columns].agg(_series_mode)

    def _apply_customer_profiles(self, data: pd.DataFrame) -> pd.DataFrame:
        if "Customer_ID" not in data.columns:
            return data

        if not self.customer_numeric_profiles_.empty:
            for column in self.customer_numeric_profiles_.columns:
                if column in data.columns:
                    profile_values = data["Customer_ID"].map(self.customer_numeric_profiles_[column])
                    data[column] = data[column].fillna(profile_values)

        if not self.customer_categorical_profiles_.empty:
            for column in self.customer_categorical_profiles_.columns:
                if column in data.columns:
                    profile_values = data["Customer_ID"].map(self.customer_categorical_profiles_[column])
                    data[column] = data[column].fillna(profile_values)

        return data

    def _add_loan_features(self, data: pd.DataFrame) -> pd.DataFrame:
        if "Type_of_Loan" not in data.columns:
            data["Loan_Type_Count"] = 0.0
            for loan_type in self.loan_types_:
                data[f"Loan_Type__{loan_type}"] = 0.0
            return data

        parsed = data["Type_of_Loan"].apply(parse_loan_types)
        data["Loan_Type_Count"] = parsed.map(len).astype("float64")
        loan_sets = parsed.map(set)
        for loan_type in self.loan_types_:
            data[f"Loan_Type__{loan_type}"] = loan_sets.map(lambda loans: float(loan_type in loans))
        return data

    def _numeric_feature(self, data: pd.DataFrame, column: str) -> pd.Series:
        if column not in data.columns:
            return pd.Series(np.nan, index=data.index, dtype="float64")
        return pd.to_numeric(data[column], errors="coerce").astype("float64")

    @staticmethod
    def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
        denominator = denominator.where(denominator.ne(0))
        result = numerator.divide(denominator)
        return result.replace([np.inf, -np.inf], np.nan).astype("float64")

    @staticmethod
    def _sum_known(*series_list: pd.Series) -> pd.Series:
        values = pd.concat(series_list, axis=1)
        result = values.fillna(0).sum(axis=1)
        return result.where(values.notna().any(axis=1)).astype("float64")

    def _add_credit_engineering_features(self, data: pd.DataFrame) -> pd.DataFrame:
        annual_income = self._numeric_feature(data, "Annual_Income")
        monthly_salary = self._numeric_feature(data, "Monthly_Inhand_Salary")
        bank_accounts = self._numeric_feature(data, "Num_Bank_Accounts")
        credit_cards = self._numeric_feature(data, "Num_Credit_Card")
        loans = self._numeric_feature(data, "Num_of_Loan")
        delayed_payments = self._numeric_feature(data, "Num_of_Delayed_Payment")
        credit_inquiries = self._numeric_feature(data, "Num_Credit_Inquiries")
        outstanding_debt = self._numeric_feature(data, "Outstanding_Debt")
        credit_history_months = self._numeric_feature(data, "Credit_History_Age_Months")
        total_emi = self._numeric_feature(data, "Total_EMI_per_month")
        monthly_investment = self._numeric_feature(data, "Amount_invested_monthly")
        monthly_balance = self._numeric_feature(data, "Monthly_Balance")
        delay_from_due_date = self._numeric_feature(data, "Delay_from_due_date")
        utilization = self._numeric_feature(data, "Credit_Utilization_Ratio")
        loan_type_count = self._numeric_feature(data, "Loan_Type_Count")

        total_credit_products = self._sum_known(credit_cards, loans)

        data["debt_to_annual_income"] = self._safe_divide(outstanding_debt, annual_income)
        data["debt_to_monthly_salary"] = self._safe_divide(outstanding_debt, monthly_salary)
        data["emi_to_monthly_salary"] = self._safe_divide(total_emi, monthly_salary)
        data["investment_to_monthly_salary"] = self._safe_divide(monthly_investment, monthly_salary)
        data["balance_to_monthly_salary"] = self._safe_divide(monthly_balance, monthly_salary)
        data["available_income_after_emi"] = monthly_salary.sub(total_emi).astype("float64")
        data["available_income_after_emi_and_investment"] = (
            monthly_salary.sub(total_emi).sub(monthly_investment).astype("float64")
        )
        data["delayed_payment_ratio"] = self._safe_divide(
            delayed_payments,
            credit_history_months,
        )
        data["avg_delay_per_delayed_payment"] = self._safe_divide(
            delay_from_due_date,
            delayed_payments,
        )
        data["inquiries_per_credit_account"] = self._safe_divide(
            credit_inquiries,
            total_credit_products,
        )
        data["credit_cards_per_bank_account"] = self._safe_divide(
            credit_cards,
            bank_accounts,
        )
        data["loans_per_bank_account"] = self._safe_divide(loans, bank_accounts)
        data["credit_history_years"] = credit_history_months.div(12).astype("float64")
        data["credit_age_per_loan"] = self._safe_divide(credit_history_months, loans)
        data["total_credit_products"] = total_credit_products
        data["loan_diversity_ratio"] = self._safe_divide(loan_type_count, loans)

        has_payment_data = delayed_payments.notna() | delay_from_due_date.notna()
        has_negative_history = delayed_payments.gt(0) | delay_from_due_date.gt(0)
        data["has_negative_payment_history"] = (
            has_negative_history.astype("float64").where(has_payment_data)
        )
        data["high_utilization_flag"] = (
            utilization.ge(35).astype("float64").where(utilization.notna())
        )
        data["low_balance_flag"] = (
            monthly_balance.le(monthly_salary.mul(0.1))
            .astype("float64")
            .where(monthly_balance.notna() & monthly_salary.notna())
        )
        data["debt_per_credit_product"] = self._safe_divide(
            outstanding_debt,
            total_credit_products,
        )

        for column in ENGINEERED_FEATURES:
            data[column] = (
                pd.to_numeric(data[column], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float64")
            )
        return data

    def _clip_numeric_features(self, data: pd.DataFrame) -> pd.DataFrame:
        for column, (lower, upper) in self.clip_bounds_.items():
            if column in data.columns:
                data[column] = pd.to_numeric(data[column], errors="coerce").clip(lower, upper)
        return data

    def _post_clean(self, data: pd.DataFrame) -> pd.DataFrame:
        data = data.drop(columns=[column for column in DROP_AFTER_CLEANING if column in data.columns])

        for column in data.columns:
            if pd.api.types.is_string_dtype(data[column]) or data[column].dtype == object:
                data[column] = data[column].astype("object").where(data[column].notna(), np.nan)

        return data

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

column_processor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, make_column_selector(dtype_include=np.number)),
        (
            "cat",
            categorical_pipeline,
            make_column_selector(dtype_include=["object", "string", "category"]),
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
column_processor.set_output(transform="pandas")

preprocessing_pipeline = Pipeline(
    steps=[
        ("cleaner", CreditScoreCleaner(use_customer_profiles=True)),
        ("columns", column_processor),
    ]
)

In [ ]:
def split_features_target(data: pd.DataFrame, target: str = TARGET) -> tuple[pd.DataFrame, pd.Series]:
    X = data.drop(columns=[target])
    y = data[target].map(SCORE_MAP)

    if y.isna().any():
        unknown_values = sorted(data.loc[y.isna(), target].dropna().unique())
        raise ValueError(f"Неизвестные значения целевой переменной: {unknown_values}")

    return X, y.astype("int64")


X, y = split_features_target(train_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_prepared = preprocessing_pipeline.fit_transform(X_train, y_train)
X_test_prepared = preprocessing_pipeline.transform(X_test)

engineered_feature_columns = [
    column for column in ENGINEERED_FEATURES if column in X_train_prepared.columns
]

print("X_train_prepared:", X_train_prepared.shape)
print("X_test_prepared:", X_test_prepared.shape)
print("Пропусков после preprocessing:", int(X_train_prepared.isna().sum().sum()))
print("Инженерных признаков:", len(engineered_feature_columns))
print("Примеры инженерных признаков:", engineered_feature_columns[:8])
X_train_prepared.head()

In [ ]:
feature_names = preprocessing_pipeline.named_steps["columns"].get_feature_names_out()
feature_names[:40], len(feature_names)

In [ ]:
prepared_diagnostics = pd.DataFrame(
    {
        "признак": X_train_prepared.columns,
        "dtype": X_train_prepared.dtypes.astype(str),
        "пропуски": X_train_prepared.isna().sum().to_numpy(),
    }
)

prepared_diagnostics.sort_values(["пропуски", "признак"], ascending=[False, True]).head(20)

## Как переиспользовать

Замените `train_raw` на DataFrame с такой же raw-схемой, затем вызовите:

```python
X_new_prepared = preprocessing_pipeline.transform(X_new)
```

Для обучения модели подгоняйте preprocessing pipeline только на train-части:

```python
X_train_prepared = preprocessing_pipeline.fit_transform(X_train, y_train)
X_valid_prepared = preprocessing_pipeline.transform(X_valid)
```

Если нужно сохранить обученный preprocessor:

```python
import joblib
joblib.dump(preprocessing_pipeline, PROJECT_ROOT / "model" / "preprocessing_pipeline.joblib")
```

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_prepared.to_csv(
    OUTPUT_DIR / "X_train_prepared.csv",
    index=True,
    index_label="source_index",
)
X_test_prepared.to_csv(
    OUTPUT_DIR / "X_test_prepared.csv",
    index=True,
    index_label="source_index",
)
y_train.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_train.csv",
    index=True,
    index_label="source_index",
)
y_test.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_test.csv",
    index=True,
    index_label="source_index",
)

print("Файлы сохранены в:", OUTPUT_DIR)
for file_name in [
    "X_train_prepared.csv",
    "X_test_prepared.csv",
    "y_train.csv",
    "y_test.csv",
]:
    print(OUTPUT_DIR / file_name)